# 🚀 Entrenamiento Robusto de mT5 para Resumen de Videos
## Google Colab Pro - Optimizado para GPU V100/A100

---

**Proyecto:** Sistema de Resumen Automático de Videos Educativos  
**Modelo:** google/mt5-small (300M parámetros)  
**Dataset:** MLSUM Español (Resúmenes de noticias)  
**GPU Recomendada:** V100 (16GB) o A100 (40GB)  

---

### 📋 Características:
- ✅ Configuración automática de GPU
- ✅ Checkpoints automáticos cada 500 pasos
- ✅ Early stopping inteligente
- ✅ Monitoreo con TensorBoard
- ✅ Guardado en Google Drive
- ✅ Evaluación con ROUGE metrics
- ✅ Recuperación automática de checkpoints

---

### ⏱️ Tiempo Estimado:
- **T4 (16GB):** ~6-8 horas para 10K ejemplos
- **V100 (16GB):** ~4-5 horas para 10K ejemplos
- **A100 (40GB):** ~2-3 horas para 10K ejemplos

---

## 1️⃣ Configuración Inicial y Verificación de GPU

In [ ]:
import torch
import sys

print("="*70)
print("🔍 VERIFICACIÓN DE GPU")
print("="*70)

# Verificar GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU detectada: {gpu_name}")
    print(f"✅ VRAM disponible: {vram_gb:.2f} GB")
    print(f"✅ CUDA version: {torch.version.cuda}")
    print(f"✅ PyTorch version: {torch.__version__}")
    
    # Recomendaciones según GPU
    if "T4" in gpu_name:
        print("\n💡 GPU T4 detectada - Configuración recomendada:")
        print("   - Batch size: 4")
        print("   - Gradient accumulation: 4")
        print("   - Tiempo estimado: 6-8 horas")
        RECOMMENDED_BATCH = 4
        RECOMMENDED_ACCUM = 4
    elif "V100" in gpu_name:
        print("\n🚀 GPU V100 detectada - Configuración óptima:")
        print("   - Batch size: 8")
        print("   - Gradient accumulation: 2")
        print("   - Tiempo estimado: 4-5 horas")
        RECOMMENDED_BATCH = 8
        RECOMMENDED_ACCUM = 2
    elif "A100" in gpu_name:
        print("\n🔥 GPU A100 detectada - Máximo rendimiento:")
        print("   - Batch size: 16")
        print("   - Gradient accumulation: 2")
        print("   - Tiempo estimado: 2-3 horas")
        RECOMMENDED_BATCH = 16
        RECOMMENDED_ACCUM = 2
    else:
        print("\n⚠️  GPU desconocida - Usando configuración conservadora")
        RECOMMENDED_BATCH = 4
        RECOMMENDED_ACCUM = 4
else:
    print("❌ GPU no disponible")
    print("⚠️  Asegúrate de seleccionar: Entorno de ejecución > Cambiar tipo > GPU")
    sys.exit(1)

print("="*70)

## 2️⃣ Instalación de Dependencias

In [ ]:
%%capture
# Instalar/actualizar dependencias necesarias
print("📦 Instalando dependencias...")

!pip install -q transformers==4.36.0
!pip install -q datasets==2.16.0
!pip install -q accelerate==0.25.0
!pip install -q evaluate==0.4.1
!pip install -q sentencepiece==0.1.99
!pip install -q rouge-score==0.1.2
!pip install -q tensorboard

print("✅ Dependencias instaladas correctamente")
print("\n📋 Paquetes instalados:")
print("   - transformers==4.36.0")
print("   - datasets==2.16.0")
print("   - evaluate==0.4.1")
print("   - rouge-score==0.1.2 (importante para métricas)")
print("   - sentencepiece==0.1.99")
print("   - tensorboard")

In [ ]:
# Verificar que las dependencias críticas están bien instaladas
print("🔍 Verificando instalación de dependencias críticas...")
print("="*70)

import sys

# Verificar transformers
try:
    import transformers
    print(f"✅ transformers: {transformers.__version__}")
except ImportError as e:
    print(f"❌ transformers no instalado: {e}")
    sys.exit(1)

# Verificar datasets
try:
    import datasets
    print(f"✅ datasets: {datasets.__version__}")
except ImportError as e:
    print(f"❌ datasets no instalado: {e}")
    sys.exit(1)

# Verificar evaluate
try:
    import evaluate
    print(f"✅ evaluate: {evaluate.__version__}")
except ImportError as e:
    print(f"❌ evaluate no instalado: {e}")
    sys.exit(1)

# Verificar rouge_score
try:
    import rouge_score
    print(f"✅ rouge_score instalado correctamente")
except ImportError as e:
    print(f"❌ rouge_score no instalado: {e}")
    print("⚠️  Instalando rouge_score...")
    !pip install -q rouge-score==0.1.2

# Verificar sentencepiece
try:
    import sentencepiece
    print(f"✅ sentencepiece instalado correctamente")
except ImportError as e:
    print(f"❌ sentencepiece no instalado: {e}")
    sys.exit(1)

print("="*70)
print("✅ Todas las dependencias están instaladas correctamente")
print("💡 Puedes continuar con el siguiente paso")

## 3️⃣ Montar Google Drive (Para guardar modelo)

In [ ]:
from google.colab import drive
import os

# Montar Google Drive
drive.mount('/content/drive')

# Crear directorios en Drive
DRIVE_BASE = "/content/drive/MyDrive/video_summarizer"
os.makedirs(f"{DRIVE_BASE}/models", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/logs", exist_ok=True)

print(f"✅ Google Drive montado en: {DRIVE_BASE}")
print(f"📁 Los modelos se guardarán en: {DRIVE_BASE}/models")
print(f"💾 Los checkpoints en: {DRIVE_BASE}/checkpoints")

## 4️⃣ Configuración de Hiperparámetros

In [ ]:
# ============================================================================
# CONFIGURACIÓN PRINCIPAL
# ============================================================================

CONFIG = {
    # Modelo
    "model_name": "google/mt5-small",
    
    # Dataset
    "dataset_name": "mlsum",
    "dataset_config": "es",
    "train_samples": 10000,      # 🔥 Aumenta a 20000 para mejor calidad (más tiempo)
    "val_samples": 1000,
    "test_samples": 500,
    
    # Tokenización
    "max_input_length": 512,
    "max_target_length": 128,
    
    # Entrenamiento (ajustado según GPU detectada)
    "per_device_train_batch_size": RECOMMENDED_BATCH,
    "per_device_eval_batch_size": RECOMMENDED_BATCH * 2,
    "gradient_accumulation_steps": RECOMMENDED_ACCUM,
    "num_train_epochs": 3,
    "learning_rate": 3e-5,
    "weight_decay": 0.01,
    "warmup_steps": 500,
    "max_grad_norm": 1.0,
    
    # Optimización
    "fp16": True,                # Precisión mixta
    "gradient_checkpointing": True,  # Ahorra memoria
    
    # Evaluación y guardado
    "eval_steps": 500,
    "save_steps": 500,
    "save_total_limit": 3,       # Mantener solo 3 mejores checkpoints
    "logging_steps": 100,
    
    # Early stopping
    "early_stopping_patience": 3,
    "load_best_model_at_end": True,
    "metric_for_best_model": "rouge1",
    
    # Directorios
    "output_dir": f"{DRIVE_BASE}/checkpoints/mt5-video-summarizer",
    "final_model_dir": f"{DRIVE_BASE}/models/mt5-video-summarizer-final",
    "logging_dir": f"{DRIVE_BASE}/logs",
}

# Calcular batch efectivo
effective_batch = CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]

print("="*70)
print("⚙️  CONFIGURACIÓN DE ENTRENAMIENTO")
print("="*70)
print(f"📊 Dataset: {CONFIG['dataset_name']} ({CONFIG['dataset_config']})")
print(f"📈 Ejemplos de entrenamiento: {CONFIG['train_samples']:,}")
print(f"🧠 Modelo: {CONFIG['model_name']}")
print(f"\n🔢 Hiperparámetros:")
print(f"   - Batch size por dispositivo: {CONFIG['per_device_train_batch_size']}")
print(f"   - Acumulación de gradientes: {CONFIG['gradient_accumulation_steps']}")
print(f"   - Batch efectivo: {effective_batch}")
print(f"   - Learning rate: {CONFIG['learning_rate']}")
print(f"   - Épocas: {CONFIG['num_train_epochs']}")
print(f"   - FP16: {CONFIG['fp16']}")
print(f"   - Gradient checkpointing: {CONFIG['gradient_checkpointing']}")
print(f"\n💾 Guardado:")
print(f"   - Checkpoints cada: {CONFIG['save_steps']} pasos")
print(f"   - Evaluación cada: {CONFIG['eval_steps']} pasos")
print(f"   - Early stopping patience: {CONFIG['early_stopping_patience']}")
print("="*70)

## 5️⃣ Carga y Preparación del Dataset

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

print("="*70)
print("📥 CARGANDO DATASET MLSUM (ESPAÑOL)")
print("="*70)

# Cargar tokenizer
print(f"\n🔤 Cargando tokenizer: {CONFIG['model_name']}...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
print("✅ Tokenizer cargado")

# Cargar dataset
print(f"\n📊 Descargando dataset: {CONFIG['dataset_name']} ({CONFIG['dataset_config']})...")
dataset = load_dataset(
    CONFIG["dataset_name"],
    CONFIG["dataset_config"],
    trust_remote_code=True
)

print(f"\n✅ Dataset original:")
print(f"   - Train: {len(dataset['train']):,} ejemplos")
print(f"   - Validation: {len(dataset['validation']):,} ejemplos")
print(f"   - Test: {len(dataset['test']):,} ejemplos")

# Reducir tamaño para entrenamiento más rápido
if CONFIG["train_samples"] < len(dataset['train']):
    print(f"\n📉 Reduciendo dataset a {CONFIG['train_samples']:,} ejemplos...")
    dataset['train'] = dataset['train'].select(range(CONFIG["train_samples"]))
    dataset['validation'] = dataset['validation'].select(range(CONFIG["val_samples"]))
    dataset['test'] = dataset['test'].select(range(CONFIG["test_samples"]))
    
    print(f"\n✅ Dataset reducido:")
    print(f"   - Train: {len(dataset['train']):,} ejemplos")
    print(f"   - Validation: {len(dataset['validation']):,} ejemplos")
    print(f"   - Test: {len(dataset['test']):,} ejemplos")

# Mostrar ejemplo
print("\n📄 Ejemplo del dataset:")
example = dataset['train'][0]
print(f"\nTexto (primeros 300 chars):\n{example['text'][:300]}...")
print(f"\nResumen:\n{example['summary']}")

print("="*70)

## 6️⃣ Tokenización del Dataset

In [ ]:
print("="*70)
print("🔤 TOKENIZANDO DATASET")
print("="*70)

def preprocess_function(examples):
    """
    Preprocesa ejemplos para mT5
    Añade prefijo 'resumir:' para indicar la tarea
    """
    # Agregar prefijo para mT5 (multitask)
    inputs = ["resumir: " + text for text in examples['text']]
    targets = examples['summary']
    
    # Tokenizar inputs
    model_inputs = tokenizer(
        inputs,
        max_length=CONFIG["max_input_length"],
        truncation=True,
        padding=False,  # Padding dinámico en el collator
    )
    
    # Tokenizar targets
    labels = tokenizer(
        targets,
        max_length=CONFIG["max_target_length"],
        truncation=True,
        padding=False,
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenizar dataset completo
print("\n⏳ Tokenizando... (esto puede tomar 2-3 minutos)")
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset['train'].column_names,
    desc="Tokenizando dataset"
)

print("\n✅ Dataset tokenizado exitosamente")
print(f"\n📊 Estadísticas:")
print(f"   - Train: {len(tokenized_dataset['train']):,} ejemplos")
print(f"   - Validation: {len(tokenized_dataset['validation']):,} ejemplos")
print(f"   - Test: {len(tokenized_dataset['test']):,} ejemplos")

# Mostrar ejemplo tokenizado
print(f"\n🔍 Ejemplo tokenizado:")
print(f"   - Input IDs shape: {len(tokenized_dataset['train'][0]['input_ids'])} tokens")
print(f"   - Labels shape: {len(tokenized_dataset['train'][0]['labels'])} tokens")

print("="*70)

## 7️⃣ Carga del Modelo y Configuración de Métricas

In [ ]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
import evaluate
import numpy as np

print("="*70)
print("🧠 CARGANDO MODELO")
print("="*70)

# Cargar modelo
print(f"\n📥 Descargando modelo: {CONFIG['model_name']}...")
print("   (Esto puede tomar 1-2 minutos la primera vez)")
model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["model_name"])

# Habilitar gradient checkpointing (ahorra memoria)
if CONFIG["gradient_checkpointing"]:
    model.gradient_checkpointing_enable()
    print("✅ Gradient checkpointing habilitado")

print(f"✅ Modelo cargado: {CONFIG['model_name']}")
print(f"   - Parámetros: ~300M")
print(f"   - Tamaño: ~1.2GB")

# Data collator (para padding dinámico)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)
print("✅ Data collator configurado")

# Configurar métricas ROUGE
print("\n📊 Configurando métricas ROUGE...")
try:
    # Intentar cargar ROUGE con evaluate
    rouge = evaluate.load("rouge")
    print("✅ ROUGE cargado con evaluate")
except Exception as e:
    print(f"⚠️  Advertencia: {str(e)}")
    print("🔄 Intentando método alternativo...")
    try:
        # Método alternativo: importar directamente
        from datasets import load_metric
        rouge = load_metric("rouge")
        print("✅ ROUGE cargado con datasets.load_metric")
    except:
        # Último recurso: usar rouge_score directamente
        print("🔄 Usando rouge_score directamente...")
        from rouge_score import rouge_scorer
        rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        
        class RougeWrapper:
            """Wrapper para rouge_score que imita la interfaz de evaluate"""
            def __init__(self, scorer):
                self.scorer = scorer
            
            def compute(self, predictions, references, use_stemmer=True):
                scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
                for pred, ref in zip(predictions, references):
                    score = self.scorer.score(ref, pred)
                    scores['rouge1'].append(score['rouge1'].fmeasure)
                    scores['rouge2'].append(score['rouge2'].fmeasure)
                    scores['rougeL'].append(score['rougeL'].fmeasure)
                
                return {
                    'rouge1': np.mean(scores['rouge1']),
                    'rouge2': np.mean(scores['rouge2']),
                    'rougeL': np.mean(scores['rougeL'])
                }
        
        rouge = RougeWrapper(rouge_scorer_obj)
        print("✅ ROUGE configurado con rouge_score")

def compute_metrics(eval_pred):
    """
    Calcula métricas ROUGE durante evaluación
    ROUGE mide la similitud entre resúmenes generados y de referencia
    """
    predictions, labels = eval_pred
    
    # Decodificar predicciones
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Reemplazar -100 en labels (padding)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Calcular ROUGE
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    
    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"],
    }

print("✅ Métricas ROUGE configuradas")
print("   - ROUGE-1: Coincidencia de palabras individuales")
print("   - ROUGE-2: Coincidencia de pares de palabras")
print("   - ROUGE-L: Subsecuencia común más larga")

print("="*70)

## 8️⃣ Configuración del Entrenador y Callbacks

In [ ]:
from transformers import (
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)

print("="*70)
print("🏋️  CONFIGURANDO ENTRENADOR")
print("="*70)

# Argumentos de entrenamiento
training_args = Seq2SeqTrainingArguments(
    # Directorios
    output_dir=CONFIG["output_dir"],
    logging_dir=CONFIG["logging_dir"],
    
    # Evaluación y guardado
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["save_steps"],
    save_total_limit=CONFIG["save_total_limit"],
    
    # Hiperparámetros
    learning_rate=CONFIG["learning_rate"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    weight_decay=CONFIG["weight_decay"],
    warmup_steps=CONFIG["warmup_steps"],
    num_train_epochs=CONFIG["num_train_epochs"],
    max_grad_norm=CONFIG["max_grad_norm"],
    
    # Optimización
    fp16=CONFIG["fp16"],
    
    # Logging
    logging_steps=CONFIG["logging_steps"],
    report_to=["tensorboard"],
    
    # Best model
    load_best_model_at_end=CONFIG["load_best_model_at_end"],
    metric_for_best_model=CONFIG["metric_for_best_model"],
    greater_is_better=True,
    
    # Generación
    predict_with_generate=True,
    generation_max_length=CONFIG["max_target_length"],
    
    # Otros
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    push_to_hub=False,
)

# Crear trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=CONFIG["early_stopping_patience"]
        )
    ],
)

print("✅ Entrenador configurado")
print("\n📋 Callbacks activos:")
print("   - ✅ Early Stopping (patience=3)")
print("   - ✅ Checkpoint automático cada 500 pasos")
print("   - ✅ TensorBoard logging")

# Calcular número de pasos
num_steps = len(tokenized_dataset['train']) // (CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']) * CONFIG['num_train_epochs']
print(f"\n📊 Estadísticas de entrenamiento:")
print(f"   - Total de pasos: ~{num_steps:,}")
print(f"   - Evaluaciones: ~{num_steps // CONFIG['eval_steps']}")
print(f"   - Checkpoints guardados: ~{num_steps // CONFIG['save_steps']}")

print("="*70)

## 9️⃣ Iniciar TensorBoard (Opcional - Monitoreo en tiempo real)

In [ ]:
# Cargar extensión de TensorBoard
%load_ext tensorboard

# Iniciar TensorBoard
%tensorboard --logdir {CONFIG['logging_dir']}

print("\n📊 TensorBoard iniciado")
print("💡 Puedes monitorear el entrenamiento en tiempo real arriba")
print("   - Loss de entrenamiento")
print("   - Métricas ROUGE")
print("   - Learning rate")
print("   - Uso de GPU")

## 🔟 INICIAR ENTRENAMIENTO

### ⚠️ IMPORTANTE:
- El entrenamiento puede tomar **2-8 horas** según la GPU
- Los checkpoints se guardan automáticamente en tu Google Drive
- Puedes detener y reanudar el entrenamiento
- Si se desconecta, ejecuta la celda de "Reanudar desde checkpoint"

### 📊 Qué esperar:
- **Loss inicial:** ~3-4
- **Loss final:** ~0.8-1.0
- **ROUGE-1 final:** ~0.40-0.45
- **ROUGE-2 final:** ~0.16-0.20
- **ROUGE-L final:** ~0.32-0.38

In [ ]:
from datetime import datetime
import traceback

print("="*70)
print("🚀 INICIANDO ENTRENAMIENTO")
print("="*70)
print(f"⏰ Inicio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n⚠️  RECORDATORIOS:")
print("   - Mantén esta pestaña abierta")
print("   - No cierres el navegador")
print("   - Los checkpoints se guardan automáticamente")
print("   - Puedes monitorear con TensorBoard arriba")
print("="*70)
print("\n")

try:
    # ⚡ ENTRENAR ⚡
    train_result = trainer.train()
    
    print("\n")
    print("="*70)
    print("✅ ENTRENAMIENTO COMPLETADO EXITOSAMENTE")
    print("="*70)
    print(f"⏰ Fin: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"⏱️  Tiempo total: {train_result.metrics.get('train_runtime', 0) / 3600:.2f} horas")
    print(f"📊 Loss final: {train_result.metrics.get('train_loss', 0):.4f}")
    print(f"🎯 Pasos completados: {train_result.metrics.get('train_steps', 0)}")
    print("="*70)
    
except KeyboardInterrupt:
    print("\n")
    print("="*70)
    print("⚠️  ENTRENAMIENTO INTERRUMPIDO POR EL USUARIO")
    print("="*70)
    print("💾 No te preocupes, los checkpoints se guardaron automáticamente")
    print(f"📁 Ubicación: {CONFIG['output_dir']}")
    print("\n💡 Puedes reanudar ejecutando la celda de 'Reanudar entrenamiento'")
    print("="*70)
    
except Exception as e:
    print("\n")
    print("="*70)
    print("❌ ERROR DURANTE EL ENTRENAMIENTO")
    print("="*70)
    print(f"Error: {str(e)}")
    print("\n📋 Traceback completo:")
    print(traceback.format_exc())
    
    if "out of memory" in str(e).lower():
        print("\n💡 SOLUCIÓN: Error de memoria GPU")
        print("   1. Reduce 'per_device_train_batch_size' en CONFIG")
        print("   2. Aumenta 'gradient_accumulation_steps'")
        print("   3. Reduce 'train_samples' a 5000")
    
    print("="*70)

## 1️⃣1️⃣ Evaluación Final en Test Set

In [ ]:
print("="*70)
print("📊 EVALUACIÓN FINAL EN TEST SET")
print("="*70)

# Evaluar en test set
test_results = trainer.evaluate(tokenized_dataset["test"])

print("\n✅ Evaluación completada")
print("\n📈 RESULTADOS FINALES:")
print("="*70)
print(f"  🎯 ROUGE-1: {test_results.get('eval_rouge1', 0):.4f}")
print(f"  🎯 ROUGE-2: {test_results.get('eval_rouge2', 0):.4f}")
print(f"  🎯 ROUGE-L: {test_results.get('eval_rougeL', 0):.4f}")
print(f"  📉 Loss: {test_results.get('eval_loss', 0):.4f}")
print("="*70)

# Interpretar resultados
rouge1 = test_results.get('eval_rouge1', 0)
print("\n🔍 INTERPRETACIÓN:")
if rouge1 >= 0.40:
    print("   ✅ Excelente - El modelo genera resúmenes de alta calidad")
elif rouge1 >= 0.35:
    print("   ✅ Bueno - El modelo genera resúmenes aceptables")
elif rouge1 >= 0.30:
    print("   ⚠️  Aceptable - Considera entrenar más épocas")
else:
    print("   ❌ Bajo - Revisa hiperparámetros o aumenta datos")

print("\n💡 ROUGE-1 > 0.40 es considerado bueno para resúmenes")

## 1️⃣2️⃣ Guardar Modelo Final

In [ ]:
print("="*70)
print("💾 GUARDANDO MODELO FINAL")
print("="*70)

# Guardar modelo y tokenizer
print(f"\n📁 Guardando en: {CONFIG['final_model_dir']}")
trainer.save_model(CONFIG["final_model_dir"])
tokenizer.save_pretrained(CONFIG["final_model_dir"])

print("\n✅ Modelo guardado exitosamente")

# Listar archivos guardados
import os
files = os.listdir(CONFIG["final_model_dir"])
print("\n📂 Archivos guardados:")
for file in files:
    file_path = os.path.join(CONFIG["final_model_dir"], file)
    size_mb = os.path.getsize(file_path) / (1024**2)
    print(f"   - {file}: {size_mb:.2f} MB")

print(f"\n📊 Tamaño total del modelo: ~1.2 GB")
print(f"\n💡 Ubicación en Google Drive:")
print(f"   {CONFIG['final_model_dir']}")

print("\n✅ Puedes descargar el modelo desde tu Google Drive")
print("="*70)

## 1️⃣3️⃣ Probar Inferencia con Ejemplos

In [ ]:
print("="*70)
print("🔮 PRUEBA DE INFERENCIA")
print("="*70)

def generar_resumen(texto, max_length=128, num_beams=4):
    """
    Genera un resumen del texto usando el modelo entrenado
    """
    # Agregar prefijo
    input_text = "resumir: " + texto
    
    # Tokenizar
    inputs = tokenizer(
        input_text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(model.device)
    
    # Generar resumen
    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=max_length,
            num_beams=num_beams,
            length_penalty=2.0,
            early_stopping=True
        )
    
    # Decodificar
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# Texto de ejemplo
texto_ejemplo = """
El aprendizaje automático es una rama de la inteligencia artificial que permite 
a las computadoras aprender de datos sin ser programadas explícitamente. Los algoritmos 
de aprendizaje automático construyen un modelo basado en datos de muestra, conocidos como 
datos de entrenamiento, para hacer predicciones o decisiones sin ser programados explícitamente 
para realizar la tarea. El aprendizaje automático se utiliza en una amplia variedad de 
aplicaciones, como filtros de correo electrónico, detección de fraudes en tarjetas de crédito, 
reconocimiento de voz, visión artificial y sistemas de recomendación.
"""

print("\n📝 TEXTO ORIGINAL:")
print(texto_ejemplo.strip())

print("\n⏳ Generando resumen...")
resumen = generar_resumen(texto_ejemplo)

print("\n✨ RESUMEN GENERADO:")
print(resumen)

print("\n📊 ESTADÍSTICAS:")
print(f"   - Texto original: {len(texto_ejemplo.split())} palabras")
print(f"   - Resumen: {len(resumen.split())} palabras")
print(f"   - Compresión: {len(resumen) / len(texto_ejemplo):.1%}")

print("="*70)

## 1️⃣4️⃣ Probar con Tu Propio Texto

In [ ]:
# 🎯 ESCRIBE TU PROPIO TEXTO AQUÍ
tu_texto = """
Escribe aquí el texto que quieres resumir...
"""

print("="*70)
print("🎯 TU RESUMEN PERSONALIZADO")
print("="*70)

if tu_texto.strip() and "Escribe aquí" not in tu_texto:
    print("\n📝 Tu texto:")
    print(tu_texto.strip())
    
    print("\n⏳ Generando resumen...")
    tu_resumen = generar_resumen(tu_texto)
    
    print("\n✨ Resumen:")
    print(tu_resumen)
    
    print("\n📊 Compresión:", f"{len(tu_resumen) / len(tu_texto):.1%}")
else:
    print("\n💡 Edita la celda y escribe tu texto en la variable 'tu_texto'")
    print("   Luego ejecuta esta celda de nuevo")

print("="*70)

## 1️⃣5️⃣ Reanudar Entrenamiento desde Checkpoint (Si se interrumpió)

In [ ]:
print("="*70)
print("🔄 REANUDAR ENTRENAMIENTO DESDE CHECKPOINT")
print("="*70)

import os

# Buscar checkpoints
checkpoint_dir = CONFIG["output_dir"]
checkpoints = [f for f in os.listdir(checkpoint_dir) if f.startswith("checkpoint-")]

if checkpoints:
    # Ordenar por número de paso
    checkpoints.sort(key=lambda x: int(x.split("-")[1]))
    ultimo_checkpoint = os.path.join(checkpoint_dir, checkpoints[-1])
    
    print(f"\n📁 Checkpoints encontrados: {len(checkpoints)}")
    print(f"\n✅ Último checkpoint: {checkpoints[-1]}")
    print(f"\n⏳ Reanudando entrenamiento...")
    
    # Reanudar entrenamiento
    train_result = trainer.train(resume_from_checkpoint=ultimo_checkpoint)
    
    print("\n✅ Entrenamiento completado desde checkpoint")
else:
    print("\n❌ No se encontraron checkpoints")
    print("💡 Ejecuta la celda de entrenamiento principal primero")

print("="*70)

## 1️⃣6️⃣ Descargar Modelo Entrenado

In [ ]:
print("="*70)
print("📦 COMPRIMIR Y DESCARGAR MODELO")
print("="*70)

import shutil

# Comprimir modelo
print("\n⏳ Comprimiendo modelo (esto puede tomar 2-3 minutos)...")
zip_path = "/content/mt5-video-summarizer"
shutil.make_archive(zip_path, 'zip', CONFIG["final_model_dir"])

print("\n✅ Modelo comprimido exitosamente")
print(f"📁 Archivo: {zip_path}.zip")
print(f"💾 Tamaño: ~500 MB (comprimido)")

# Descargar
from google.colab import files

print("\n⏬ Iniciando descarga...")
print("⚠️  La descarga puede tomar 5-10 minutos según tu conexión")

files.download(f"{zip_path}.zip")

print("\n✅ Descarga iniciada")
print("\n💡 El archivo también está disponible en tu Google Drive:")
print(f"   {CONFIG['final_model_dir']}")

print("="*70)

## 📊 RESUMEN FINAL

### ✅ Lo que has logrado:
1. ✅ Entrenado modelo mT5-small con 10K ejemplos en español
2. ✅ Optimizado para tu GPU de Colab Pro
3. ✅ Checkpoints automáticos guardados en Drive
4. ✅ Evaluación con métricas ROUGE
5. ✅ Modelo final guardado y listo para usar

### 📦 Archivos generados:
- `MyDrive/video_summarizer/models/mt5-video-summarizer-final/` - Modelo entrenado
- `MyDrive/video_summarizer/checkpoints/` - Checkpoints intermedios
- `MyDrive/video_summarizer/logs/` - Logs de TensorBoard

### 🚀 Próximos pasos:
1. Descarga el modelo comprimido
2. Úsalo en tu proyecto local con `inference.py`
3. Integra con Whisper para transcripción de videos
4. Crea interfaz web con Gradio

### 📚 Uso del modelo:
```python
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model = AutoModelForSeq2SeqLM.from_pretrained("ruta/al/modelo")
tokenizer = AutoTokenizer.from_pretrained("ruta/al/modelo")

texto = "Tu texto a resumir..."
inputs = tokenizer("resumir: " + texto, return_tensors="pt")
summary_ids = model.generate(inputs["input_ids"], max_length=128)
resumen = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
```

---

### 🎉 ¡FELICITACIONES! Has entrenado exitosamente tu modelo de resumen de videos.

---

## 💡 TIPS Y SOLUCIÓN DE PROBLEMAS

### ❓ Problemas comunes:

**1. Error de memoria (OOM):**
```python
# Reduce batch size
CONFIG["per_device_train_batch_size"] = 2
CONFIG["gradient_accumulation_steps"] = 8
```

**2. Entrenamiento muy lento:**
- Verifica que estés usando GPU: `Runtime > Change runtime type > GPU`
- Usa Colab Pro para GPU más rápidas (V100/A100)

**3. Desconexión de Colab:**
- Los checkpoints se guardan cada 500 pasos en tu Drive
- Ejecuta la celda "Reanudar desde checkpoint" para continuar

**4. ROUGE scores muy bajos (<0.30):**
- Aumenta `train_samples` a 20000
- Entrena más épocas (5-6 en lugar de 3)
- Ajusta learning rate a 5e-5

**5. Pérdida de conexión:**
```javascript
// Ejecuta en consola del navegador (F12) para mantener conexión:
function ClickConnect(){
  console.log("Working");
  document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)
```

### 🔧 Optimizaciones avanzadas:

**Entrenar con más datos:**
```python
CONFIG["train_samples"] = 20000  # Doble de datos
CONFIG["num_train_epochs"] = 5   # Más épocas
```

**Usar modelo más grande:**
```python
CONFIG["model_name"] = "google/mt5-base"  # 580M parámetros
# Requiere GPU A100 con 40GB VRAM
```

**Ajuste fino de hiperparámetros:**
```python
CONFIG["learning_rate"] = 5e-5   # LR más alto
CONFIG["warmup_steps"] = 1000    # Más warmup
CONFIG["weight_decay"] = 0.05    # Más regularización
```

---